# CSA — same checkpoint131: greedy / beam5 / EOS

GPU + Internet ON. Add `csa-real-v126-lemma-fixed` and `csa-semantic-banks-v130`.
Enable Kaggle Secret `KAGGLE_API_TOKEN` (or edit SECRET_NAME to `codex`). Downloads ONLY checkpoint.bin from exact Output131, verifies full SHA.
No training, recache, minimumlength, lengthpenalty or changed weights. Same64val, original beam5replay plus greedy/EOS trace. GPU0 worker cap15min is notETA.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
COMMIT='165dd753fde22746df3a2fd06824a3b6de720218'
HASHES={'real_bank_data.py': '00e7110105ec98a5ef41969f9e51074815a42a9b69deac90b2614e9679d2bcf9', 'data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'REAL_BANK_INPUT_PACKAGE.json': 'c60902e04ea15cf90f652e384937bb317e288753f13b8ede6ab5ef89f8d97129', 'PRE_FULL_CHECK_PROTOCOL.md': '3e11fb2dde0e28c2b744a469bdf52e850f01c3696fddad4e78fed0e0eec716f3', 'run_decode_check.py': '7215a21cbe7d7cfc148b98ac78291a9eb837ba8075a660a07c2fc9976f3ac565', 'model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'pilot_data.py': '134cc6de37a6a2afd6d9d242ea74edddde08eb3f2fd90771e34c779ae104f820', 'SAVE_BANK_V130_PACKAGE.json': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1', 'xe_pilot_review_text/input_receipt.json': 'cf6f9e963c6cb4b79ac2e5c6f203cadd52706c3d05609656121586f846ce1165', 'xe_pilot_review_text/captions_final.json': '21a419bf872447f0a13c648834e4d35a30582a3068bfcd1fc3fdaefeb720f337'}
OUTPUT=Path('/kaggle/working/csa_decode_check_v131')
PREPARED_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-real-v126-lemma-fixed')
BANK_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
if OUTPUT.exists():raise RuntimeError('Preserve existing output; use a fresh session.')
if shutil.disk_usage('/kaggle/working').free<2*1024**3:raise RuntimeError('Insufficient free disk; preserve existing files.')
OUTPUT.mkdir(parents=True)
SOURCE=Path(tempfile.mkdtemp(prefix='csa_decode_check_v131-'))
LOG=OUTPUT/'notebook.log'
for n,h in HASHES.items():
    data=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{n}',timeout=90).read()
    if hashlib.sha256(data).hexdigest()!=h:raise ValueError('Changed source: '+n)
    target=SOURCE/n;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(data)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES),indent=2),encoding='utf-8')
print('Source pinned:',COMMIT,flush=True)


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Worker active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Worker wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    API_SETUP='import os,re,subprocess,sys\nfrom kaggle_secrets import UserSecretsClient\nif KERNEL_VERSION!=\'ducanh2403/2-vi-cai-kia-qua-nhieu/131\':raise ValueError(\'This notebook pins131; do not substitute latest or another version.\')\nif not re.fullmatch(r\'[a-zA-Z0-9_-]+\',OWNER) or not re.fullmatch(r\'[a-z0-9]+(?:-[a-z0-9]+)*\',SLUG):raise ValueError(\'Invalid OWNER/SLUG\')\nif not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError(\'TITLE/SLUG must have6–50characters\')\ntry:\n    token=UserSecretsClient().get_secret(SECRET_NAME).strip()\n    if not token:raise ValueError(\'empty\')\nexcept Exception:raise RuntimeError(\'Enable the configured SECRET_NAME in Add-ons > Secrets. Do not paste token in code/chat.\') from None\ndef run_cli(args,authenticated=True):\n    environment=os.environ.copy()\n    for key in (\'KAGGLE_API_TOKEN\',\'KAGGLE_USERNAME\',\'KAGGLE_KEY\'):environment.pop(key,None)\n    if authenticated:environment[\'KAGGLE_API_TOKEN\']=token\n    environment[\'PYTHONUNBUFFERED\']=\'1\'\n    with subprocess.Popen(args,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors=\'replace\') as process:\n        for line in process.stdout:print(line.replace(token,\'[REDACTED]\'),end=\'\',flush=True)\n        code=process.wait()\n    if code:raise RuntimeError(f\'Command failed ({code}); see redacted output. No automatic fallback or dataset overwrite.\')\n\nrun_cli([sys.executable,\'-m\',\'pip\',\'install\',\'--upgrade\',\'--force-reinstall\',\'--quiet\',\'https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip\'],authenticated=False)\nVERSION_PREFLIGHT="import tempfile\\nfrom types import SimpleNamespace\\nfrom unittest.mock import patch\\nfrom kaggle.api.kaggle_api_extended import KaggleApi\\n\\n# Exercise the installed download implementation without contacting Kaggle.\\napi = KaggleApi()\\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, \'build_kaggle_client\') as factory:\\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\\n    api.kernels_output(\'ducanh2403/2-vi-cai-kia-qua-nhieu/131\', path=directory)\\n    request = endpoint.call_args.args[0]\\n    assert getattr(request, \'version_label\', None) == \'v131\', \'Installed CLI ignores the requested version; STOP before download.\'\\n    assert request.user_name == \'ducanh2403\'\\n    assert request.kernel_slug == \'2-vi-cai-kia-qua-nhieu\'\\nprint(\'Version routing preflight PASS: request.version_label=v131; no network call.\', flush=True)\\n"\nrun_cli([sys.executable,\'-u\',\'-c\',VERSION_PREFLIGHT])\n'

    KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/131'
    OWNER='ducanh2403'
    SLUG='csa-decode-check'
    TITLE='CSA decode check'
    SECRET_NAME='KAGGLE_API_TOKEN'  # Change to codex if that is your enabled secret name.
    exec(compile(API_SETUP,'pinned_cli_setup.py','exec'))
    DOWNLOAD=SOURCE/'checkpoint-download';DOWNLOAD.mkdir()
    run_cli([sys.executable,'-m','kaggle','kernels','output',KERNEL_VERSION,'-p',str(DOWNLOAD),
             '--file-pattern',r'(^|/)checkpoint\.bin$'])
    found=list(DOWNLOAD.rglob('checkpoint.bin'))
    if len(found)!=1:raise RuntimeError('Expected exactly one Output131 checkpoint; no latest fallback.')
    h=hashlib.sha256()
    with found[0].open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    if h.hexdigest()!='99b1f9936b864e549182f2307c369a221e4ca4f17b0f0cad9c60f3fb1cb7017e':raise ValueError('Output131 checkpoint SHA differs')
    (OUTPUT/'download_receipt.json').write_text(json.dumps(dict(kernel_version=KERNEL_VERSION,checkpoint_sha256=h.hexdigest()),indent=2),encoding='utf-8')
    run([sys.executable,'-u',SOURCE/'run_decode_check.py','--checkpoint',found[0],
         '--prepared-root',PREPARED_ROOT,'--bank-root',BANK_ROOT,'--output',OUTPUT],timeout=900)

    complete=True
except Exception:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8')
    raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,training_executed=False)),encoding='utf-8')
    review=Path(str(OUTPUT)+'_review.zip')
    globals_only={'source_receipt.json','input_receipt.json','progress.json','manifest.json','receipt.json',
                  'throughput.json','failure.txt','notebook.log','run_status.json'}
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.iterdir()):
            if p.is_file() and p.suffix in ('.json','.log','.txt') and ('decode'=='decode' or p.name in globals_only):z.write(p,p.name)
    print('SEND REVIEW:',review,flush=True)
    print('KEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('COMPLETE' if complete else 'INCOMPLETE; preserve completed artifacts/logs.',flush=True)
